# Análise Steam — Arquitetura Medalhão (Bronze → Silver → Gold)

Este notebook organiza os dados da Steam em camadas seguindo a **Arquitetura de Medalhão**:

| Camada | Objetivo | Tabela(s) |
|--------|----------|----------|
| **Bronze** | Dados brutos, conforme ingeridos | `workspace.default.steam_games` |
| **Silver** | Dados limpos, filtrados e normalizados | `workspace.default.steam_games_silver`, `workspace.default.steam_generos_silver` |
| **Gold** | Agregações prontas para análise — uma por pergunta | Ver tabela abaixo |

### Perguntas respondidas na camada Gold:

| # | Pergunta | Tabela(s) Gold |
|---|----------|----------------|
| 1 | Existe diferença de aceitação entre jogos gratuitos e pagos? | `steam_gold_monetizacao` |
| 2 | Qual a faixa de preço médio nas categorias mais bem avaliadas? | `steam_gold_preco_genero` |
| 3 | Como evoluiu o volume anual de lançamentos? | `steam_gold_lancamentos` |
| 4 | De que forma a presença de conquistas (achievements) impacta a taxa de aprovação e média de horas jogadas? | `steam_gold_achievements_engajamento` |
| 5 | Quais títulos têm as maiores médias de tempo de jogo? | `steam_gold_top_playtime_titles` |
| 6 | Quais gêneros concentram as maiores médias de horas jogadas? | `steam_gold_playtime_genero` |
| 7 | Como evoluiu o preço médio dos jogos ao longo dos anos? | `steam_gold_evolucao_precos` |

---

## 🔶 Camada Bronze — Dados Brutos

A tabela `workspace.default.steam_games` contém os dados brutos importados via upload de arquivo, sem nenhuma transformação. Esta é a base de toda a pipeline.

In [0]:
%sql
-- ===== BRONZE: TABELA BRUTA DE JOGOS DA STEAM =====
-- Dados brutos conforme ingeridos via upload de arquivo
-- Nenhuma transformação aplicada nesta camada

-- Contagem total de registros brutos
SELECT COUNT(*) AS total_registros_brutos FROM workspace.default.steam_games;

-- Amostra dos dados brutos
SELECT * FROM workspace.default.steam_games;

---

## ⚪ Camada Silver — Dados Limpos e Normalizados

A camada Silver recebe os dados brutos da Bronze e aplica limpeza, filtros de qualidade e normalização.

Tabelas criadas:
- `steam_games_silver` — uma linha por jogo, com colunas limpas, derivadas e `faixa_achievements` (classificação de conquistas)
- `steam_generos_silver` — uma linha por (jogo, gênero), com parsing do array JSON e EXPLODE

In [0]:
%sql
-- ===== SILVER: TABELA PRINCIPAL DE JOGOS LIMPOS =====
-- Cria a tabela silver a nível de jogo (uma linha por jogo)
-- 
-- Transformações aplicadas:
--   • Classifica modelo de monetizacao (Gratuito vs Pago)
--   • Filtra apenas jogos com pelo menos uma avaliacao (qualidade)
--   • Filtra jogos sem data de lancamento
--   • Remove aplicativos de software do catálogo (Audio Production, Video Production, etc.)
--   • Remove gênero 'Free To Play' (gênero está errado no dataset)
--   • Mantem colunas relevantes para analise downstream

CREATE OR REPLACE TABLE workspace.default.steam_games_silver AS
SELECT 
    app_id,
    name,
    release_date,
    YEAR(release_date) AS ano_lancamento,
    price,
    price_status,
    CASE 
        WHEN price = 0 OR LOWER(price_status) = 'free' THEN 'Gratuito'
        ELSE 'Pago'
    END AS modelo_monetizacao,
    estimated_owners,
    genres,
    categories,
    positive,
    negative,
    positive + negative AS total_avaliacoes,
    ROUND((positive * 100.0) / NULLIF(positive + negative, 0), 2) AS taxa_aprovacao_pct,
    recommendations,
    peak_ccu,
    metacritic_score,
    user_score,
    average_playtime_forever,
    median_playtime_forever,
    achievements,
    dlc_count,
    windows,
    mac,
    linux,
    developers,
    publishers,
    CASE 
        WHEN COALESCE(achievements, 0) = 0 THEN '0 (sem conquistas)'
        WHEN achievements BETWEEN 1 AND 10 THEN '1-10'
        WHEN achievements BETWEEN 11 AND 25 THEN '11-25'
        WHEN achievements BETWEEN 26 AND 50 THEN '26-50'
        WHEN achievements BETWEEN 51 AND 100 THEN '51-100'
        WHEN achievements > 100 THEN '100+'
    END AS faixa_achievements
FROM workspace.default.steam_games
WHERE (positive + negative) > 0  -- Apenas jogos com avaliacoes
  AND release_date IS NOT NULL
  -- Remove aplicativos de software do catálogo
  AND genres NOT LIKE '%Audio Production%'
  AND genres NOT LIKE '%Video Production%'
  AND genres NOT LIKE '%Utilities%'
  AND genres NOT LIKE '%Web Publishing%'
  AND genres NOT LIKE '%Software Training%'
  AND genres NOT LIKE '%Animation & Modeling%'
  AND genres NOT LIKE '%Photo Editing%'
  AND genres NOT LIKE '%Design & Illustration%'
  AND genres NOT LIKE '%Game Development%'
  -- Remove gênero 'Free To Play' (gênero está errado no dataset)
  AND genres NOT LIKE '%Free To Play%';

-- Amostra dos dados criados na camada Silver
SELECT * FROM workspace.default.steam_games_silver LIMIT 10;

In [0]:
%sql
-- ===== SILVER: TABELA DE GÊNEROS NORMALIZADA =====
-- Cria a tabela silver com gêneros explodidos (uma linha por jogo-gênero)
--
-- Transformações aplicadas:
--   • Faz o parsing do array JSON de gêneros
--   • Usa EXPLODE para criar uma linha por gênero
--   • Trata os dois formatos (string simples e objeto JSON)
--   • Filtra gêneros vazios

CREATE OR REPLACE TABLE workspace.default.steam_generos_silver AS
SELECT 
    s.app_id,
    s.name,
    s.price,
    s.modelo_monetizacao,
    s.positive,
    s.negative,
    s.total_avaliacoes,
    COALESCE(
        get_json_object(exploded_genre, '$.description'),
        exploded_genre
    ) AS genero
FROM workspace.default.steam_games_silver s
LATERAL VIEW EXPLODE(from_json(s.genres, 'array<string>')) AS exploded_genre
WHERE s.genres IS NOT NULL
  AND COALESCE(
        get_json_object(exploded_genre, '$.description'),
        exploded_genre
    ) IS NOT NULL
  AND LENGTH(COALESCE(
        get_json_object(exploded_genre, '$.description'),
        exploded_genre
    )) > 0;

-- Amostra dos dados criados na camada Silver
SELECT * FROM workspace.default.steam_generos_silver LIMIT 10;

---

## 🟡 Camada Gold — Agregações por Pergunta de Negócio

Cada tabela Gold responde a uma pergunta específica, construída a partir da camada Silver e persistida no Unity Catalog.

In [0]:
%sql
-- ===== GOLD Q1: MODELO DE MONETIZAÇÃO x APROVAÇÃO =====
-- Agrega a taxa de aprovação por modelo de monetizacao (Gratuito vs Pago)
--
-- Fonte: workspace.default.steam_games_silver
-- Responde: Existe diferenca de aceitacao entre jogos gratuitos e pagos?

CREATE OR REPLACE TABLE workspace.default.steam_gold_monetizacao AS
SELECT 
    modelo_monetizacao,
    COUNT(*) AS total_titulos,
    SUM(positive) AS soma_positivas,
    SUM(negative) AS soma_negativas,
    ROUND((SUM(positive) * 100.0) / NULLIF(SUM(positive) + SUM(negative), 0), 2) AS taxa_aprovacao_pct
FROM workspace.default.steam_games_silver
GROUP BY modelo_monetizacao
ORDER BY modelo_monetizacao;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_monetizacao;

In [0]:
%sql
-- ===== GOLD Q2: PREÇO MÉDIO POR GÊNERO (MELHORES AVALIADOS) =====
-- Agrega preco medio e taxa de aprovacao por genero
-- Filtra apenas generos com volume significativo (>= 50 jogos) e jogos pagos
--
-- Fonte: workspace.default.steam_generos_silver
-- Responde: Qual a faixa de preco medio nas categorias mais bem avaliadas?

CREATE OR REPLACE TABLE workspace.default.steam_gold_preco_genero AS
SELECT 
    genero,
    COUNT(*) AS total_jogos,
    ROUND((SUM(positive) * 100.0) / NULLIF(SUM(positive) + SUM(negative), 0), 2) AS taxa_aprovacao_pct,
    ROUND(AVG(price), 2) AS preco_medio,
    ROUND(MIN(price), 2) AS preco_minimo,
    ROUND(MAX(price), 2) AS preco_maximo
FROM workspace.default.steam_generos_silver
WHERE price > 0  -- Apenas jogos pagos para analise de preco
  AND genero != 'Free To Play' --Genero está errado no dataset
GROUP BY genero
HAVING COUNT(*) >= 50  -- Volume significativo
ORDER BY taxa_aprovacao_pct DESC
LIMIT 15;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_preco_genero;

In [0]:
%sql
-- ===== GOLD Q3: EVOLUÇÃO ANUAL DE LANÇAMENTOS =====
-- Agrega o volume de lancamentos por ano, separando pagos e gratuitos
--
-- Fonte: workspace.default.steam_games_silver
-- Responde: Como evoluiu o volume anual de novos lancamentos?

CREATE OR REPLACE TABLE workspace.default.steam_gold_lancamentos AS
SELECT 
    ano_lancamento,
    COUNT(DISTINCT app_id) AS total_lancamentos,
    COUNT(DISTINCT CASE WHEN price > 0 THEN app_id END) AS jogos_pagos,
    COUNT(DISTINCT CASE WHEN price = 0 THEN app_id END) AS jogos_gratuitos,
    ROUND(AVG(price), 2) AS preco_medio_lancamentos
FROM workspace.default.steam_games_silver
WHERE ano_lancamento >= 2012
  AND ano_lancamento <= 2026
GROUP BY ano_lancamento
ORDER BY ano_lancamento ASC;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_lancamentos;

---

### Q4: Achievements x Engajamento

**Pergunta:** De que forma a presença de conquistas (achievements) impacta a taxa de aprovação e média de horas jogadas?

**Fonte:** `workspace.default.steam_games_silver`
**Proxies de engajamento:** playtime médio, recomendações, peak CCU, total de avaliações

Tabela:
- `steam_gold_achievements_engajamento` — engajamento por faixa de conquistas (classificação `faixa_achievements` da Silver)

In [0]:
%sql
-- ===== GOLD Q4: IMPACTO DOS ACHIEVEMENTS NO ENGAJAMENTO =====
-- Engajamento por faixa de conquistas (classificação vinda da Silver)
--
-- Fonte: workspace.default.steam_games_silver (coluna faixa_achievements)
-- Responde: De que forma a presença de conquistas impacta a taxa de aprovação e média de horas jogadas?
CREATE OR REPLACE TABLE workspace.default.steam_gold_achievements_engajamento AS
SELECT 
    faixa_achievements,
    COUNT(*) AS total_jogos,
    ROUND(AVG(average_playtime_forever), 2) AS avg_playtime_minutos,
    ROUND(AVG(median_playtime_forever), 2) AS median_playtime_minutos,
    ROUND(AVG(recommendations), 2) AS avg_recomendacoes,
    ROUND(AVG(peak_ccu), 2) AS avg_peak_ccu,
    ROUND(AVG(total_avaliacoes), 2) AS avg_total_avaliacoes,
    ROUND(AVG(taxa_aprovacao_pct), 2) AS avg_taxa_aprovacao
FROM workspace.default.steam_games_silver
GROUP BY faixa_achievements
ORDER BY MIN(COALESCE(achievements, 0));

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_achievements_engajamento;

---

### Q5: Títulos com Maiores Médias de Tempo de Jogo

**Pergunta:** Quais são os títulos com as maiores médias históricas de tempo de jogo?

**Fonte:** `workspace.default.steam_games_silver`

Tabela:
- `steam_gold_top_playtime_titles` — top 20 títulos por tempo médio de jogo

In [0]:
%sql
-- ===== GOLD Q5: TOP TÍTULOS POR TEMPO MÉDIO DE JOGO =====
-- Ranking dos 20 jogos com maior média histórica de tempo de jogo
--
-- Fonte: workspace.default.steam_games_silver
-- Responde: Quais são os títulos com as maiores médias de tempo de jogo?

CREATE OR REPLACE TABLE workspace.default.steam_gold_top_playtime_titles AS
SELECT 
    name,
    app_id,
    average_playtime_forever AS avg_playtime_minutos,
    median_playtime_forever AS median_playtime_minutos,
    modelo_monetizacao,
    taxa_aprovacao_pct,
    total_avaliacoes
FROM workspace.default.steam_games_silver
ORDER BY average_playtime_forever DESC
LIMIT 20;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_top_playtime_titles;

---

### Q6: Gêneros com Maiores Médias de Tempo de Jogo

**Pergunta:** Quais gêneros concentram as maiores médias de horas jogadas por usuário?

**Fonte:** `workspace.default.steam_generos_silver` + `workspace.default.steam_games_silver`

Tabela:
- `steam_gold_playtime_genero` — média de horas jogadas por gênero

In [0]:
%sql
-- ===== GOLD Q6: GÊNEROS COM MAIORES MÉDIAS DE TEMPO DE JOGO =====
-- Agrega a média de tempo de jogo por gênero
--
-- Fonte: workspace.default.steam_generos_silver + workspace.default.steam_games_silver
-- Responde: Quais gêneros concentram as maiores médias de horas jogadas?

CREATE OR REPLACE TABLE workspace.default.steam_gold_playtime_genero AS
SELECT 
    g.genero,
    COUNT(*) AS total_jogos,
    ROUND(AVG(s.average_playtime_forever), 2) AS avg_playtime_minutos,
    ROUND(AVG(s.median_playtime_forever), 2) AS median_playtime_minutos,
    ROUND(AVG(s.total_avaliacoes), 2) AS avg_avaliacoes
FROM workspace.default.steam_generos_silver g
JOIN workspace.default.steam_games_silver s ON g.app_id = s.app_id
GROUP BY g.genero
HAVING COUNT(*) >= 50
ORDER BY avg_playtime_minutos DESC;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_playtime_genero;

---

### Q7: Evolução de Preços ao Longo dos Anos

**Pergunta:** Como evoluiu o preço médio dos jogos ao longo dos anos? Existe tendência de valorização ou desvalorização?

**Fonte:** `workspace.default.steam_games_silver`

Tabela:
- `steam_gold_evolucao_precos` — preço médio, mediana e distribuição por ano de lançamento

In [0]:
%sql
-- ===== GOLD Q7: EVOLUÇÃO DE PREÇOS AO LONGO DOS ANOS =====
-- Analisa a evolução do preço médio dos jogos por ano de lançamento
--
-- Fonte: workspace.default.steam_games_silver
-- Responde: Como evoluiu o preço médio dos jogos ao longo dos anos?

CREATE OR REPLACE TABLE workspace.default.steam_gold_evolucao_precos AS
SELECT 
    ano_lancamento,
    COUNT(*) AS total_jogos_pagos,
    ROUND(AVG(price), 2) AS preco_medio,
    ROUND(PERCENTILE(price, 0.5), 2) AS preco_mediana,
    ROUND(MIN(price), 2) AS preco_minimo,
    ROUND(MAX(price), 2) AS preco_maximo,
    ROUND(STDDEV(price), 2) AS desvio_padrao_preco
FROM workspace.default.steam_games_silver
WHERE price > 0  -- Apenas jogos pagos
  AND ano_lancamento >= 2012
  AND ano_lancamento <= 2026
GROUP BY ano_lancamento
ORDER BY ano_lancamento ASC;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_evolucao_precos;